# 03 — Economic surprises and headline tone

Two more "does this make sense?" signals:

1. **Data surprises** — when a release (CPI, payrolls...) comes in far
   from what economists expected. Available on ANY Terminal via normal
   reference fields, so this always works.
2. **Headline tone** — a simple positive/negative word score on news
   headlines. Bloomberg *news* via API needs a separate entitlement
   (EID) that many desks don't have, so this part is built to degrade
   gracefully: it runs on sample headlines unless you wire in a real feed.

### How the surprise score works

For each economic ticker Bloomberg stores, for the latest release:

- `ACTUAL_RELEASE`   — the printed number
- `BN_SURVEY_MEDIAN` — economists' median forecast
- `BN_SURVEY_HIGH` / `BN_SURVEY_LOW` — the most extreme forecasts

We score:  `surprise = (actual - median) / (high - low)`

Dividing by the survey *range* normalizes across series with different
units (CPI in %, payrolls in thousands). Interpretation:
**|score| > 0.5 means the print landed OUTSIDE the entire forecast range**
— a genuine shock. |score| < 0.2 is a yawn.

In [1]:
import pandas as pd

import bbg
import db

# US economic release tickers (type ECST <GO> on the Terminal to find more)
ECO_TICKERS = [
    "NFP TCH Index",    # Nonfarm payrolls, monthly change
    "CPI YOY Index",    # CPI, year-over-year
    "CPUPXCHG Index",   # Core CPI, month-over-month
    "GDP CQOQ Index",   # GDP, quarter-over-quarter annualized
    "INJCJC Index",     # Initial jobless claims (weekly)
    "CONSSENT Index",   # University of Michigan sentiment
    "NAPMPMI Index",    # ISM manufacturing PMI
    "RSTAMOM Index",    # Retail sales, month-over-month
]

FIELDS = ["ACTUAL_RELEASE", "BN_SURVEY_MEDIAN",
          "BN_SURVEY_HIGH", "BN_SURVEY_LOW", "ECO_RELEASE_DT"]

data = bbg.bdp(ECO_TICKERS, FIELDS)
print(data)

                ACTUAL_RELEASE  BN_SURVEY_MEDIAN  BN_SURVEY_HIGH  \
NFP TCH Index              3.4               3.3             3.8   
CPI YOY Index              3.6               4.0             4.5   
CPUPXCHG Index             1.5               1.9             2.4   
GDP CQOQ Index             1.0               0.5             1.0   
INJCJC Index               3.9               4.1             4.6   
CONSSENT Index             2.0               1.9             2.4   
NAPMPMI Index              1.2               1.6             2.1   
RSTAMOM Index              3.1               2.8             3.3   

                BN_SURVEY_LOW ECO_RELEASE_DT  
NFP TCH Index             2.8       20260709  
CPI YOY Index             3.5       20260709  
CPUPXCHG Index            1.4       20260709  
GDP CQOQ Index            0.0       20260709  
INJCJC Index              3.6       20260709  
CONSSENT Index            1.4       20260709  
NAPMPMI Index             1.1       20260709  
RSTAMOM Ind

### Score each release and store it

In [2]:
conn = db.get_connection()
rows = []

for ticker in data.index:
    actual = data.loc[ticker, "ACTUAL_RELEASE"]
    median = data.loc[ticker, "BN_SURVEY_MEDIAN"]
    high = data.loc[ticker, "BN_SURVEY_HIGH"]
    low = data.loc[ticker, "BN_SURVEY_LOW"]
    release_dt = data.loc[ticker, "ECO_RELEASE_DT"]

    # skip if Bloomberg didn't return everything (no survey, not released yet)
    if actual is None or median is None or high is None or low is None:
        continue
    if high == low:  # avoid dividing by zero when all forecasts agree
        continue

    score = (actual - median) / (high - low)
    rows.append({"ticker": ticker, "release": release_dt,
                 "actual": actual, "survey": median,
                 "surprise": round(score, 2)})
    db.insert_surprise(conn, ticker, release_dt, actual, median, round(score, 3))

surprises = pd.DataFrame(rows).sort_values("surprise", key=abs, ascending=False)
print(surprises.to_string(index=False))
print("\n|surprise| > 0.5 = outside the whole forecast range (a real shock)")

        ticker  release  actual  survey  surprise
GDP CQOQ Index 20260709     1.0     0.5       0.5
 CPI YOY Index 20260709     3.6     4.0      -0.4
CPUPXCHG Index 20260709     1.5     1.9      -0.4
 NAPMPMI Index 20260709     1.2     1.6      -0.4
 RSTAMOM Index 20260709     3.1     2.8       0.3
  INJCJC Index 20260709     3.9     4.1      -0.2
 NFP TCH Index 20260709     3.4     3.3       0.1
CONSSENT Index 20260709     2.0     1.9       0.1

|surprise| > 0.5 = outside the whole forecast range (a real shock)


### Headline tone (simple on purpose)

Fancy sentiment models exist, but a word-list scorer is transparent,
fast, and you can see exactly WHY a headline scored the way it did —
which matters when you're deciding whether to trust an alert.

Score = (positive words - negative words) / total words in the headline.

In [3]:
POSITIVE_WORDS = ["beat", "beats", "surge", "surges", "rally", "rallies",
                  "gain", "gains", "strong", "record", "upgrade", "eases",
                  "improves", "optimism", "soars", "rebound", "cools"]

NEGATIVE_WORDS = ["miss", "misses", "plunge", "plunges", "slump", "slumps",
                  "fall", "falls", "weak", "downgrade", "fears", "warns",
                  "crisis", "recession", "default", "selloff", "spikes",
                  "cuts", "tumbles", "jitters", "shock"]


def tone_score(headline):
    """Return a tone score between -1 (very negative) and +1 (very positive)."""
    words = headline.lower().replace(",", " ").split()
    positives = sum(1 for w in words if w in POSITIVE_WORDS)
    negatives = sum(1 for w in words if w in NEGATIVE_WORDS)
    if len(words) == 0:
        return 0.0
    return (positives - negatives) / len(words)


# Demo headlines. To use REAL Bloomberg news you need the news API
# entitlement — see the markdown cell below. Until then, paste headlines
# here or wire in another source.
sample_headlines = [
    "Stocks surge as CPI cools more than expected",
    "Treasury yields spike on hot payrolls, stocks tumble",
    "Credit markets shrug off equity selloff in unusual divergence",
    "Fed warns of persistent inflation risks, markets slump",
]

for h in sample_headlines:
    print(f"{tone_score(h):+.3f}  {h}")

+0.250  Stocks surge as CPI cools more than expected
+0.000  Treasury yields spike on hot payrolls, stocks tumble
-0.111  Credit markets shrug off equity selloff in unusual divergence
-0.250  Fed warns of persistent inflation risks, markets slump


### The incoherence angle on tone

The interesting signal is not the tone itself — it is **tone vs price
action disagreeing**. Negative headlines + credit spreads tightening =
the same kind of mismatch notebook 02 hunts for. Once you have a real
headline feed, average `tone_score` per day and correlate it with SPX
moves using exactly the machinery from notebook 02.

### Getting real Bloomberg headlines (when entitled)

- The API service is `//blp/mktnews-content` (subscription-based).
  It needs a **news EID entitlement** on top of your Terminal — ask your
  Bloomberg rep; most plain Terminal seats do not have it.
- Practical no-entitlement workaround: `NI <topic> <GO>` on the Terminal,
  export headlines, and point `tone_score` at the export file.